# Notebook pulito per addestrare e salvare il modello MNIST

In [ ]:

import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras import layers


In [3]:

# Carica il training set e il test set di MNIST come coppie (immagine, etichetta)
train_ds, test_ds = tfds.load(
    "mnist",                          # Nome del dataset
    split=["train", "test"],         # Divisioni da scaricare
    as_supervised=True,               # Restituisce (immagine, label)
)


In [4]:


def prepara_immagine(img, label):
    img = tf.cast(img, tf.float32) / 255.0 # Converte i pixel in float32 e li normalizza tra 0 e 1
    return img, label



In [5]:

train_ds = train_ds.map(prepara_immagine)
test_ds = test_ds.map(prepara_immagine)
train_ds = train_ds.shuffle(10000)
train_ds = train_ds.batch(32)  #uso batch di 32 immagini
test_ds = test_ds.batch(32)
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)# Precarica i batch in memoria per rendere più veloce l'esecuzione
test_ds = test_ds.prefetch(tf.data.AUTOTUNE)


In [7]:

# Crea il modello sequenziale
model = tf.keras.Sequential([
    layers.Conv2D(32, (3, 3), activation="relu", input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2)), # Riduce la dimensione spaziale mantenendo le informazioni importanti
    layers.Conv2D(64, (3, 3), activation="relu"),# Secondo strato convoluzionale più profondo
    layers.MaxPooling2D((2, 2)),# Secondo max pooling
    layers.Flatten(),# Trasforma le mappe 2D in un vettore 1D
    layers.Dense(128, activation="relu"),# Strato denso per combinare le caratteristiche imparate
    layers.Dropout(0.3),     # Dropout per ridurre l'overfitting
    layers.Dense(10, activation="softmax"),# Strato finale con 10 uscite, una per ogni cifra da 0 a 9
])


In [8]:

# Compila il modello specificando ottimizzatore, loss e metrica
model.compile(
    optimizer="adam",                             # Algoritmo di ottimizzazione
    loss="sparse_categorical_crossentropy",       # Loss per classificazione multiclasse
    metrics=["accuracy"],                         # Metrica da monitorare
)


In [9]:

# Addestra il modello sul training set e valida sul test set
model.fit(
    train_ds,                     # Dataset di addestramento
    epochs=5,                     # Numero di epoche
    validation_data=test_ds,      # Dataset di validazione
)


Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 16s 8ms/step - accuracy: 0.9514 - loss: 0.1584 - val_accuracy: 0.9868 - val_loss: 0.0397
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 16s 8ms/step - accuracy: 0.9831 - loss: 0.0569 - val_accuracy: 0.9882 - val_loss: 0.0332
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - accuracy: 0.9876 - loss: 0.0396 - val_accuracy: 0.9879 - val_loss: 0.0369
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - accuracy: 0.9900 - loss: 0.0323 - val_accuracy: 0.9921 - val_loss: 0.0258
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - accuracy: 0.9920 - loss: 0.0242 - val_accuracy: 0.9925 - val_loss: 0.0262


In [10]:

# Valuta il modello sul test set finale
model.evaluate(test_ds)

# Salva il modello nel formato moderno Keras
model.save("modello_numeri.keras")

# Stampa un messaggio finale
print("Modello salvato come modello_numeri.keras")


313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9925 - loss: 0.0262
Modello salvato come modello_numeri.keras
